# M6 외부 검증 — V1-t 레시피 이식 (Kaggle Recruit, 38차 후속)

> 배경: 단일 매장(영업일 ~250일) 검증의 일반화 한계 보완. 실매장 추가 확보 불가·KADX 접근 불가로
> **Kaggle *Recruit Restaurant Visitor Forecasting***(일본 음식점 829곳, 2016-01~2017-04 일별 방문객)를
> 외부 검증 셋으로 채택. 작성: 2026-07-28

🔒 데이터는 Kaggle 대회 규칙 동의 후 다운로드해 `AI/data/raw/sales/kaggle/`(gitignore)에 두고 실행 —
커밋본은 관행대로 출력 제거 상태.

**사용 원칙 — 벤치마크 도전이 아니라 "우리 설계 결정의 외부 검증"**
- **동결 이식**: 서빙(`app/model/predictor.py`)과 동일 구성 — V1-t 고정 파라미터, 60라운드 고정
  (조기 종료 없음 = 매장별 적응 0), **h=1 단일 모델 + h-앵커 피처**(38차 §6 승인 전략)
- 피처는 우리 셋과의 **교집합만**(dow 원핫 7·is_holiday·lag1·roll7·lag_dow·roll4dow) — 새 피처 발명 금지.
  기온·학사·regime·roll_atv는 대응물 없어 제외
- 우리 매장 학습에 혼입하지 않음 · Recruit에서 튜닝해 역이식하지 않음 · 예약 데이터 미사용
- 프로토콜: 매장별 expanding 월 단위 walk-forward — val month = 영업일 ≥10일(우리 fold 규칙),
  본 평가는 직전 이력 ≥60 영업일(SHORT_HISTORY 임계), 10~59일 구간은 임계 검증용 별도 버킷

**사전 등록 판정 기준(결과 확인 전 고정)** — "레시피 이식성 확인" =
D+1 기준 **과반 매장에서 MA-7 대비 MAE 개선 AND 매장별 상대 개선율 중앙값 ≤ −3%**.

## 판정 요약 (TL;DR)

1. **사전 기준 충족 — 레시피 이식성 확인.** 본 평가 814개 매장에서 D+1 **승률 92.6%, 상대 MAE 중앙값
   −10.1%**(Q1 −16.5% / Q3 −5.3%, 전체 합산 −13.5%) — 우리 매장의 −10.2%와 사실상 동일한 우위가
   800개 매장 규모에서 재현됨. 파일럿 매장 특수성이 아니라 레시피의 성질.
2. **서빙 h=1 단일 모델 전략이 D+3까지 유지** — D+2 −10.3% / D+3 −10.4%(승률 93.7%/92.9%),
   계단 감쇠 없음. 달력 신호(요일·공휴일)가 지배하는 환경에서는 h-앵커의 신선도 손실이 미미 —
   38차 §6 승인의 일반화 근거.
3. **SHORT_HISTORY=60일 임계 실증** — (매장,월) 단위 이력 구간별 승률·중앙값이
   10~59일 59.1%·−1.4% → 60~119일 79.1%·−9.0% → 300일+ 89.2%·−13.1%로 **60일 경계에서 불연속
   개선 후 단조 상승**. 임계 타당 + "데이터 축적 = 최대 레버"(06 TL;DR 5) 교차 확인 —
   우리 매장(~250일)도 축적만으로 추가 이득 기대 구간.
4. **업종·규모 정합** — Izakaya(주점) 194개: 승률 95.9%·중앙값 −11.0%로 전 장르 상위권.
   방문 규모 하위 1/3(노이즈 큰 소규모 매장)도 88.8%·−7.3% — 우리 매장 조건에서도 성립.
5. **(B단계, §4) pooling은 기존 매장 무익 · cold-start엔 유효** — main: 글로벌 vs 로컬 승률 53.9%
   (기준 >55% 미달) → **매장별 fit-on-request 유지**(현 stateless 서빙 설계 지지). short(10~59일):
   글로벌 승률 72~79%·중앙값 −6~−7%(사전 기준 충족, 로컬 59%·−1.4% 대비) → 다매장 확장 시
   **"신규 매장 첫 60일 global prior → 이후 로컬 전환"** 하이브리드의 실증 근거.

**함의**: ① 단일 매장 한계 보완 완료(발표 근거) ② 다매장 확장 시 레시피 재사용 가능
③ 신규 매장 cold-start 처방 확보(§4 — Chronos zero-shot의 GBM 대안).

In [ ]:
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
BASE = AI_DIR / "data/raw/sales/kaggle/recruit-restaurant-visitor-forecasting"

PAL = {"blue": "#2a78d6", "orange": "#eb6834", "gray": "#d9d8d4", "ink2": "#52514e"}
_installed = {f.name for f in fm.fontManager.ttflist}
plt.rcParams.update({
    "font.family": [f for f in ("AppleGothic", "Apple SD Gothic Neo", "NanumGothic") if f in _installed] or ["sans-serif"],
    "axes.unicode_minus": False, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "figure.dpi": 100,
})

# 서빙(predictor.py)과 동일한 고정 구성 — 조기 종료·매장별 튜닝 없음
V1T_PARAMS = {
    "objective": "regression", "learning_rate": 0.0257, "num_leaves": 9,
    "min_data_in_leaf": 10, "feature_fraction": 0.6796, "lambda_l1": 0.001,
    "lambda_l2": 0.0, "seed": 42, "verbosity": -1,
}
N_ROUNDS = 60
WARMUP_OPEN_DAYS = 60      # SHORT_HISTORY 임계와 동일
VAL_MIN_OPEN_DAYS = 10     # 우리 fold 규칙과 동일
HS = (1, 2, 3)

visits = pd.read_csv(BASE / "air_visit_data.csv.zip", parse_dates=["visit_date"])
dates = pd.read_csv(BASE / "date_info.csv.zip", parse_dates=["calendar_date"])
stores = pd.read_csv(BASE / "air_store_info.csv.zip")
HOLIDAYS = set(dates.loc[dates.holiday_flg == 1, "calendar_date"])
GENRE = stores.set_index("air_store_id").air_genre_name
print(f"매장 {visits.air_store_id.nunique()}개, 행 {len(visits):,}, "
      f"기간 {visits.visit_date.min():%Y-%m-%d}~{visits.visit_date.max():%Y-%m-%d}")

In [ ]:
# §1 이식 루프 — 매장×월 expanding walk-forward (전체 ~9,000 fit, 로컬 ~7분)
def build_features(s: pd.Series):
    """영업일 축 h별 피처 — 06 §1 build_h와 동일 구조(교집합 피처만)."""
    idx = s.index
    static = pd.DataFrame(index=idx)
    static["is_holiday"] = [1.0 if d in HOLIDAYS else 0.0 for d in idx]
    dow = pd.get_dummies(idx.dayofweek, prefix="dow").set_axis(idx, axis=0)
    for i in range(7):
        if f"dow_{i}" not in dow:
            dow[f"dow_{i}"] = 0
    dow = dow[[f"dow_{i}" for i in range(7)]].astype(float)
    bydow = s.groupby(idx.dayofweek)
    lag_dow = bydow.shift(1)                                   # 같은 요일 직전 — h≤7 안전
    roll4dow = bydow.apply(lambda g: g.shift(1).rolling(4).mean()).droplevel(0).reindex(idx)
    out = {}
    for h in HS:
        X = pd.concat([static, dow], axis=1)
        X["lag_sales_h"] = s.shift(h)
        r7h = s.shift(h).rolling(7).mean()
        X["roll7_h"] = r7h
        X["lag_dow"] = lag_dow
        X["roll4dow"] = roll4dow
        out[h] = (X, r7h)
    return out

t0 = time.time()
recs, n_fits = [], 0
for sid, g in visits.groupby("air_store_id"):
    s = g.set_index("visit_date").visitors.sort_index().astype(float)
    n = len(s)
    if n < WARMUP_OPEN_DAYS + VAL_MIN_OPEN_DAYS:
        continue
    feats = build_features(s)
    X1, r71 = feats[1]
    y_ratio = np.log1p(s) - np.log1p(r71)
    months = pd.Series(s.index.to_period("M"), index=s.index)
    open_rank = pd.Series(np.arange(n), index=s.index)

    for m in months.unique():
        va = s.index[months == m]
        if len(va) < VAL_MIN_OPEN_DAYS:
            continue
        hist = int(open_rank[va[0]])          # val month 시작 시점의 직전 이력 영업일 수
        if hist < VAL_MIN_OPEN_DAYS:
            continue
        bucket = "main" if hist >= WARMUP_OPEN_DAYS else "short"
        tr = s.index[s.index < va[0]]
        ytr = y_ratio.loc[tr]
        k = ytr.notna()
        if int(k.sum()) < VAL_MIN_OPEN_DAYS:
            continue
        model = lgb.train(V1T_PARAMS, lgb.Dataset(X1.loc[tr][k], label=ytr[k]),
                          num_boost_round=N_ROUNDS)
        n_fits += 1
        a = s.loc[va]
        for h in HS:                          # 서빙 전략: h=1 모델을 h-앵커 피처로 재사용
            Xh, r7h = feats[h]
            p = np.expm1(model.predict(Xh.loc[va]) + np.log1p(r7h.loc[va]))
            naive = r7h.loc[va]
            ok = np.isfinite(p) & naive.notna()
            if ok.sum() == 0:
                continue
            e_m = np.abs(a[ok] - p[ok])
            e_n = np.abs(a[ok] - naive[ok])
            recs.append(dict(store=sid, month=str(m), bucket=bucket, h=h, n=int(ok.sum()),
                             hist_open=hist, sae_model=float(e_m.sum()), sae_naive=float(e_n.sum()),
                             ss_model=float((2 * e_m / (a[ok] + np.abs(p[ok]))).sum()),
                             ss_naive=float((2 * e_n / (a[ok] + naive[ok])).sum())))

res = pd.DataFrame(recs)
print(f"fits={n_fits:,}, {time.time()-t0:.0f}s | (store,month,h) 행 {len(res):,}")

In [ ]:
# §2 판정 — 매장 단위 집계(본 평가: 이력 ≥60 영업일) + 사전 등록 기준
def store_table(df, h):
    d = df[df.h == h].groupby("store")[["sae_model", "sae_naive", "n", "ss_model", "ss_naive"]].sum()
    d["mae_model"], d["mae_naive"] = d.sae_model / d.n, d.sae_naive / d.n
    d["rel"] = d.mae_model / d.mae_naive - 1
    d["smape_model"], d["smape_naive"] = d.ss_model / d.n * 100, d.ss_naive / d.n * 100
    return d

main = res[res.bucket == "main"]
rows = []
for h in HS:
    st = store_table(main, h)
    rows.append((f"D+{h}", len(st), f"{(st.rel < 0).mean():.1%}", f"{st.rel.median():+.1%}",
                 f"{st.rel.quantile(.25):+.1%} ~ {st.rel.quantile(.75):+.1%}",
                 f"{st.sae_model.sum() / st.sae_naive.sum() - 1:+.1%}",
                 f"{st.smape_model.mean():.1f}% / {st.smape_naive.mean():.1f}%"))
display(pd.DataFrame(rows, columns=["선행", "매장수", "승률(vs MA-7)", "상대 MAE 중앙값",
                                    "IQR", "전체 합산", "sMAPE 모델/naive"]))

st1 = store_table(main, 1)
crit = (st1.rel < 0).mean() > 0.5 and st1.rel.median() <= -0.03
print(f"사전 기준(D+1 과반 승 AND 중앙값 ≤ −3%): {'충족 → 레시피 이식성 확인' if crit else '미충족'}")

g1 = st1.assign(genre=st1.index.map(GENRE)).groupby("genre").agg(
    n=("rel", "size"), 승률=("rel", lambda x: (x < 0).mean()), 중앙값=("rel", "median"))
display(g1[g1.n >= 20].sort_values("n", ascending=False).round(3))

In [ ]:
# §3 SHORT_HISTORY=60 임계 실증 + 규모 층화
d1 = res[res.h == 1].copy()
d1["rel_m"] = d1.sae_model / d1.sae_naive - 1
d1["hist_bin"] = pd.cut(d1["hist_open"], [10, 60, 120, 200, 300, 500],
                        labels=["10-59", "60-119", "120-199", "200-299", "300+"], right=False)
strat = d1.groupby("hist_bin", observed=True).rel_m.agg(
    n="size", 승률=lambda x: (x < 0).mean(), 중앙값="median")
display(strat.round(3))

st = store_table(main, 1)
small = st[st.mae_naive <= st.mae_naive.quantile(0.33)]
print(f"방문 규모 하위 1/3(소규모·고노이즈) 매장: {len(small)}개 | "
      f"승률 {(small.rel < 0).mean():.1%} | 중앙값 {small.rel.median():+.1%}")

fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.4), constrained_layout=True)
ax = axes[0]
ax.bar(strat.index.astype(str), strat["승률"] * 100, color=PAL["blue"], width=0.62)
ax.axhline(50, color=PAL["ink2"], lw=1, ls="--")
ax.axvline(0.5, color=PAL["orange"], lw=1.4, ls=":")
ax.text(0.56, 95, "SHORT_HISTORY 임계(60일)", color=PAL["orange"], fontsize=8.5)
ax.set_ylabel("MA-7 대비 승률 (%)")
ax.set_title("이력 영업일 구간별 승률 — 60일 경계 불연속")
ax = axes[1]
ax.hist(st1.rel.clip(-0.5, 0.5) * 100, bins=40, color=PAL["blue"], alpha=0.85)
ax.axvline(0, color=PAL["ink2"], lw=1)
ax.axvline(st1.rel.median() * 100, color=PAL["orange"], lw=1.6, ls="--",
           label=f"중앙값 {st1.rel.median():+.1%}")
ax.set_xlabel("매장별 상대 MAE vs MA-7 (%) — 음수=모델 우위")
ax.set_title("본 평가 814개 매장 분포 (D+1)")
ax.legend(frameon=False, fontsize=9)
plt.show()

### 관찰 — 세부

- **재현의 질**: 우리 매장에서 확정한 우위 폭(D+1 −10.2%)이 814개 매장의 **중앙값(−10.1%)과 일치**.
  즉 파일럿 성적은 분포의 한가운데 — 과대평가도 요행도 아님. 분포 좌측 꼬리(Q1 −16.5%)는
  요일 패턴이 강한 매장, 우측 꼬리의 패배 매장(7.4%)은 대부분 이력이 짧거나 변동이 극단적.
- **계단이 평평한 이유**: 우리 매장은 D+1 −10.2% → D+3 −1.4%로 감쇠했지만 Recruit에선 D+3까지
  −10.4% 유지. 교집합 피처가 달력(요일·공휴일) 지배 구성이라 **타깃일 기준 피처는 선행일과 무관** —
  h-앵커로 낡는 것은 lag 계열뿐인데 그 비중이 상대적으로 작음. 함의: 신뢰도 T4(LONG_HORIZON)는
  매장·피처 구성에 따라 보수적일 수 있으나, 우리 매장 실측(−1.4%) 기준으론 유지가 정직.
- **60일 임계**: 10~59일 구간은 승률 59.1%·중앙값 −1.4%로 "naive보다 낫긴 하나 못 믿을" 수준 —
  SHORT_HISTORY 배지로 경고하는 현 설계가 정확히 맞는 처방. 60일을 넘으면 즉시 −9%대로 점프.
- **한계(정직)**: 방문객 수≠매출(우리 order_count에 대응), 일본 상권·공휴일 체계, 기온·학사 피처
  부재로 **우리 20열 전체가 아닌 골격(비율 타깃+달력+lag 구조)의 이식 검증**임. 매출 타깃·전체
  피처의 검증은 여전히 실매장 데이터만 가능.

### 다음 단계

- **B단계 완료(§4)** — B1 pooling 기각(기존 매장, 매장별 학습 유지) · B2 cold-start 근거 확보
  (신규 매장 첫 60일 global prior 하이브리드 — Phase 8+/다매장 시 활성화, 현 MVP 미적용).
- spec 반영: `model_spec.md` §3 외부 검증·cold-start 메모 갱신 (docs PR #22 동승).

In [ ]:
# §4 B단계 — 전 매장 통합 학습(pooling) + cold-start probe
# 사전 선언(탐색 금지): 글로벌 설정 2개 고정. 시간 누수 방지 — 글로벌도 월 walk-forward.
#  B1 판정: 글로벌(어느 설정이든)이 main 셀 >55%에서 로컬을 이기고 Δ중앙값<0 → "pooling 이득".
#  B2 판정: short 버킷에서 글로벌이 MA-7 대비 승률 ≥70% AND 중앙값 ≤ −5% → "cold-start 보완 가능".
FEATS = ["is_holiday"] + [f"dow_{i}" for i in range(7)] + ["lag_sales_h", "roll7_h", "lag_dow", "roll4dow"]

frames = []
for sid, g in visits.groupby("air_store_id"):
    s = g.set_index("visit_date").visitors.sort_index().astype(float)
    if len(s) < 20:
        continue
    Xf, r7f = build_features(s)[1]
    Xf = Xf.copy()
    Xf["target"] = np.log1p(s) - np.log1p(r7f)
    Xf["y"], Xf["r7"], Xf["store"] = s.values, r7f.values, sid
    Xf["rank"] = np.arange(len(s))
    frames.append(Xf.reset_index(names="date"))
G = pd.concat(frames, ignore_index=True)
G["month"] = G.date.dt.to_period("M")
print(f"글로벌 프레임 {len(G):,}행, 매장 {G.store.nunique()}개")

CONFIGS = {
    "global_v1t": (V1T_PARAMS, N_ROUNDS),  # V1-t 동결 그대로
    "global_cap": (dict(objective="regression", learning_rate=0.05, num_leaves=63,
                        min_data_in_leaf=50, feature_fraction=0.9, seed=42, verbosity=-1), 300),
}

t0 = time.time()
cells = []
for m in sorted(G.month.unique()):
    tr = G[(G.month < m) & G.target.notna()]
    if len(tr) < 1000:
        continue
    va = G[G.month == m].copy()
    for name, (params, rounds) in CONFIGS.items():
        model = lgb.train(params, lgb.Dataset(tr[FEATS], label=tr.target), num_boost_round=rounds)
        va[f"p_{name}"] = np.expm1(model.predict(va[FEATS]) + np.log1p(va.r7.values))
    for sid, grp in va.groupby("store"):
        hist = int(grp["rank"].iloc[0])
        if len(grp) < VAL_MIN_OPEN_DAYS or hist < VAL_MIN_OPEN_DAYS:  # §1과 동일한 셀 정의
            continue
        g_ = grp[grp.r7.notna()]
        if len(g_) == 0:
            continue
        row = dict(store=sid, month=str(m), n=len(g_), sae_naive=float(np.abs(g_.y - g_.r7).sum()))
        for name in CONFIGS:
            row[f"sae_{name}"] = float(np.abs(g_.y - g_[f"p_{name}"]).sum())
        cells.append(row)
print(f"글로벌 walk-forward {time.time()-t0:.0f}s | 평가 셀 {len(cells):,}")

# 로컬(§1) 결과와 페어 조인 — hist_open 동반
A1 = res[res.h == 1][["store", "month", "bucket", "hist_open", "sae_model", "n"]].rename(
    columns={"sae_model": "sae_local"})
M = pd.DataFrame(cells).merge(A1, on=["store", "month"], suffixes=("", "_a"))
assert (M.n == M.n_a).all(), "셀 정의 불일치"
for c in ["local", "global_v1t", "global_cap"]:
    M[f"rel_{c}"] = M["sae_local" if c == "local" else f"sae_{c}"] / M.sae_naive - 1

rows = []
for bucket in ["main", "short"]:
    df = M[M.bucket == bucket]
    for c in ["local", "global_v1t", "global_cap"]:
        d = df[f"rel_{c}"] - df["rel_local"]
        rows.append((bucket, c, len(df), f"{(df[f'rel_{c}'] < 0).mean():.1%}",
                     f"{df[f'rel_{c}'].median():+.1%}",
                     "—" if c == "local" else f"{(d < 0).mean():.1%} / {d.median():+.2%}"))
display(pd.DataFrame(rows, columns=["버킷", "모델", "셀수", "vs MA-7 승률", "vs MA-7 중앙값",
                                    "vs local 승률/Δ중앙값"]))

sh = M[M.bucket == "short"].copy()
sh["hist_bin"] = pd.cut(sh["hist_open"], [10, 20, 40, 60],
                        labels=["10-19", "20-39", "40-59"], right=False)
display(sh.groupby("hist_bin", observed=True)["rel_global_cap"].agg(
    n="size", 승률=lambda x: (x < 0).mean(), 중앙값="median").round(3))

print("===== 사전 선언 판정 =====")
main_b = M[M.bucket == "main"]
b1 = any(((main_b[f"rel_{c}"] - main_b.rel_local) < 0).mean() > 0.55
         and (main_b[f"rel_{c}"] - main_b.rel_local).median() < 0
         for c in ["global_v1t", "global_cap"])
print(f"B1 pooling 이득(기존 매장): {'있음' if b1 else '없음 → 매장별 fit-on-request 유지'}")
b2_pass = [c for c in ["global_v1t", "global_cap"]
           if (M[M.bucket == 'short'][f'rel_{c}'] < 0).mean() >= 0.70
           and M[M.bucket == 'short'][f'rel_{c}'].median() <= -0.05]
print(f"B2 cold-start 보완: {'가능 — ' + ', '.join(b2_pass) + ' 기준 충족' if b2_pass else '기준 미달'} "
      f"(로컬 기준선 59.1%·−1.4%)")

### §4 관찰 — B1 pooling 기각(기존 매장), B2 cold-start 근거 확보

- **B1 기각**: main 셀에서 global_cap vs local 승률 53.9%·Δ중앙값 −0.85%로 기준(>55%) 미달,
  global_v1t(저용량 글로벌)는 32.6%로 명확 열세 → **이력 있는 매장은 매장별 학습 유지**.
  현 stateless fit-on-request 서빙 설계를 지지하는 결과 — 중앙 학습·모델 배포 인프라 비용을
  정당화할 이득이 없음.
- **B2 충족**: short(10~59일) 셀에서 global_v1t **78.6%·−6.2%**, global_cap **72.0%·−7.4%**
  (로컬 59.1%·−1.4% — 둘 다 사전 기준 통과). 이력이 짧을수록 글로벌 우위가 커지는 방향성도 일관
  (40~59일 구간 76.9%·−8.7%). 단 10~19일 구간은 표본 27셀이라 판단 유보.
- **함의(Phase 8+/다매장 로드맵)**: 신규 매장 온보딩 = **첫 60일 global prior(타 매장 데이터 학습)
  서빙 → 60일 후 로컬 전환** 하이브리드의 실증 근거. 현 단일 매장 MVP엔 타 매장 데이터가 없어
  미적용 — SHORT_HISTORY 배지 유지가 현행 정답. model_spec §3의 Chronos zero-shot cold-start 메모에
  GBM global prior를 대안으로 병기.